# Sistema de Recomendação de Pets para Adoção
Pipeline: carregamento → encoding → normalização → PCA → KMeans → similaridade por cosseno

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('https://raw.githubusercontent.com/eduardapicolo/PI5-turma01-Grupo03/feature/data-extration/data/animais_adotar.csv')

keep = ['url', 'imagem', 'nome', 'raca', 'localizacao',
        'tipo_animal', 'sexo', 'porte', 'idade', 'pelagem',
        'cuidados_veterinarios', 'temperamento', 'vive_bem_com', 'sociavel_com']
df = df[[c for c in keep if c in df.columns]]
df = df[df['sexo'] != 'ambos']
df = df.dropna(subset=['tipo_animal', 'sexo', 'porte', 'idade',
                        'pelagem', 'cuidados_veterinarios',
                        'temperamento', 'vive_bem_com', 'sociavel_com'])
df = df.sample(n=min(6000, len(df)), random_state=42).reset_index(drop=True)

for col in ['tipo_animal', 'sexo', 'porte', 'idade', 'pelagem',
            'cuidados_veterinarios', 'temperamento', 'vive_bem_com', 'sociavel_com']:
    df[col] = df[col].astype(str).str.lower().str.strip()

# corrige encoding de URLs de imagem
if 'imagem' in df.columns:
    df['imagem'] = df['imagem'].astype(str).str.replace('&amp;', '&', regex=False)

df_original = df.copy()
print(f'Dataset: {len(df)} animais | {df.tipo_animal.value_counts().to_dict()}')

## Encoding e Normalização

In [ ]:
from sklearn.preprocessing import MinMaxScaler

df_enc = df.copy()

# Label encoding
df_enc['tipo_animal'] = df_enc['tipo_animal'].map({'cachorro': 0, 'gato': 1})
df_enc['sexo']        = df_enc['sexo'].map({'fêmea': 0, 'macho': 1})
df_enc['porte']       = df_enc['porte'].map({'pequeno': 0, 'médio': 1, 'grande': 2})
df_enc['idade']       = df_enc['idade'].map({
    'abaixo de 2 meses': 0, '2 a 6 meses': 1, '7 a 11 meses': 2,
    '1 ano': 3, '2 anos': 4, '3 anos': 5, '4 anos': 6,
    '5 anos': 7, '6 ou mais anos': 8
})

# One-Hot Encoding
df_pelagem      = df_enc['pelagem'].str.get_dummies(sep=', ').add_prefix('pelagem_')
df_cuidados     = df_enc['cuidados_veterinarios'].str.get_dummies(sep=', ').add_prefix('cuidados_')
df_temperamento = df_enc['temperamento'].str.get_dummies(sep=', ').add_prefix('temperamento_')
df_vive         = df_enc['vive_bem_com'].str.get_dummies(sep=', ').add_prefix('vive_')
df_sociavel     = df_enc['sociavel_com'].str.get_dummies(sep=', ').add_prefix('sociavel_')

df_feat = pd.concat([
    df_enc[['tipo_animal', 'sexo', 'porte', 'idade']],
    df_pelagem, df_cuidados, df_temperamento, df_vive, df_sociavel
], axis=1).dropna()

valid_idx   = df_feat.index
df_original = df_original.loc[valid_idx].reset_index(drop=True)
df_feat     = df_feat.reset_index(drop=True)

# Separar por tipo ANTES de normalizar (evita vazamento entre distribuições)
mask_c    = df_feat['tipo_animal'] == 0
mask_g    = df_feat['tipo_animal'] == 1
feat_cols = [c for c in df_feat.columns if c != 'tipo_animal']

df_c_raw = df_feat[mask_c][feat_cols].reset_index(drop=True)
df_g_raw = df_feat[mask_g][feat_cols].reset_index(drop=True)

orig_c = df_original[mask_c.values].reset_index(drop=True)
orig_g = df_original[mask_g.values].reset_index(drop=True)

scaler_c = MinMaxScaler()
scaler_g = MinMaxScaler()
df_c = pd.DataFrame(scaler_c.fit_transform(df_c_raw), columns=feat_cols)
df_g = pd.DataFrame(scaler_g.fit_transform(df_g_raw), columns=feat_cols)

print(f'Cachorros: {len(df_c)} | Gatos: {len(df_g)} | Features: {len(feat_cols)}')
print('\nColunas de temperamento:', [c for c in feat_cols if c.startswith('temperamento_')])
print('Colunas de sociavel:    ', [c for c in feat_cols if c.startswith('sociavel_')])

## PCA — redução de dimensionalidade (90% da variância)

In [ ]:
from sklearn.decomposition import PCA

pca_cachorro = PCA(n_components=0.9, random_state=42)
arr_c = pca_cachorro.fit_transform(df_c)
df_pca_cachorro = pd.DataFrame(arr_c, columns=[f'PC{i+1}' for i in range(arr_c.shape[1])])

pca_gato = PCA(n_components=0.9, random_state=42)
arr_g = pca_gato.fit_transform(df_g)
df_pca_gato = pd.DataFrame(arr_g, columns=[f'PC{i+1}' for i in range(arr_g.shape[1])])

print(f'PCs cachorro: {arr_c.shape[1]} | PCs gato: {arr_g.shape[1]}')

## KMeans — busca do K ótimo por silhouette score
- **Cachorros: K=6** (silhouette≈0.1642)
- **Gatos: K=7** (silhouette≈0.156)

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

def melhor_kmeans(df_pca, label):
    best_k, best_score = 2, -1
    for k in range(2, 8):
        km = KMeans(n_clusters=k, random_state=42, n_init=10)
        labels = km.fit_predict(df_pca)
        score = silhouette_score(df_pca, labels)
        print(f'  {label} k={k}  silhouette={score:.4f}')
        if score > best_score:
            best_score, best_k = score, k
    km_final = KMeans(n_clusters=best_k, random_state=42, n_init=10)
    km_final.fit(df_pca)
    print(f'→ Melhor k para {label}: {best_k} (score={best_score:.4f})\n')
    return km_final, best_k

kmeans_cachorro, k_c = melhor_kmeans(df_pca_cachorro, 'cachorro')
kmeans_gato,     k_g = melhor_kmeans(df_pca_gato,    'gato')

df_pca_cachorro['cluster'] = kmeans_cachorro.predict(df_pca_cachorro)
df_pca_gato['cluster']     = kmeans_gato.predict(df_pca_gato)

orig_c['cluster'] = df_pca_cachorro['cluster'].values
orig_g['cluster'] = df_pca_gato['cluster'].values

## Visualização dos clusters

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, df_pca, titulo in [
    (axes[0], df_pca_cachorro, f'Cachorros (k={k_c})'),
    (axes[1], df_pca_gato,     f'Gatos (k={k_g})'),
]:
    sc = ax.scatter(df_pca['PC1'], df_pca['PC2'], c=df_pca['cluster'],
                    cmap='tab10', alpha=0.5, s=8)
    ax.set_title(titulo)
    ax.set_xlabel('PC1')
    ax.set_ylabel('PC2')
    plt.colorbar(sc, ax=ax)

plt.tight_layout()
plt.show()

## Um animal representativo de cada cluster

In [ ]:
def mostrar_clusters(df_orig, label):
    print(f'\n=== {label.upper()} ===')
    for c in sorted(df_orig['cluster'].unique()):
        row = df_orig[df_orig['cluster'] == c].sample(1, random_state=42).iloc[0]
        nome = row.get('nome', '') if isinstance(row.get('nome', ''), str) else ''
        raca = row.get('raca', '') if isinstance(row.get('raca', ''), str) else ''
        print(f'  Cluster {int(c)}: {nome or "(sem nome)"} | {raca or "(sem raça)"} | '
              f'sexo={row.sexo}, porte={row.porte}, idade={row.idade}, '
              f'pelagem={row.pelagem}, temperamento={row.temperamento}, '
              f'sociavel_com={row.sociavel_com}, cuidados={row.cuidados_veterinarios}')
        tem_img = str(row.get('imagem', '')).startswith('http')
        print(f'           imagem: {"✅ " + str(row.imagem)[:60] if tem_img else "❌ sem imagem"}')

mostrar_clusters(orig_c, 'cachorro')
mostrar_clusters(orig_g, 'gato')

## Similaridade por cosseno

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

pc_cols_c = [c for c in df_pca_cachorro.columns if c != 'cluster']
pc_cols_g = [c for c in df_pca_gato.columns    if c != 'cluster']

sim_cachorro = cosine_similarity(df_pca_cachorro[pc_cols_c])
sim_gato     = cosine_similarity(df_pca_gato[pc_cols_g])

print(f'Matriz cachorro: {sim_cachorro.shape} | Matriz gato: {sim_gato.shape}')

## Função de recomendação

Parâmetros do dict `preferencias`:
| Campo | Tipo | Valores possíveis |
|---|---|---|
| `sexo` | str | `'macho'`, `'fêmea'`, `'indiferente'` |
| `porte` | str | `'pequeno'`, `'médio'`, `'grande'`, `'indiferente'` |
| `idade` | str | `'filhote'`, `'jovem'`, `'adulto'`, `'idoso'`, `'indiferente'` |
| `pelagem` | str | `'preta'`, `'branca'`, `'caramela'`, `'marrom'`, `'cinza'`, `'rajada'`, `'tricolor'` |
| `temperamento` | str | `'dócil'`, `'brincalhão'`, `'sociável'`, `'calmo'`, `'independente'`, `'indiferente'` |
| `moradia` | str | `'apartamento'`, `'casa com quintal'`, `'ambos'` |
| `criancas` | bool | `True`, `False` |
| `outros_pets` | list | `['cachorros']`, `['gatos']`, `['cachorros', 'gatos']`, `[]` |
| `cuidados` | list | `['castrado']`, `['vacinado']`, `['vermifugado']` |
| `necessidades_especiais` | bool | `True`, `False` |

In [ ]:
TEMPERAMENTO_GRUPOS = {
    'dócil':        ['temperamento_dócil'],
    'brincalhão':   ['temperamento_brincalhão'],
    'sociável':     ['temperamento_sociável'],
    'calmo':        ['temperamento_calmo'],
    'independente': ['temperamento_independente'],
}

PELAGEM_GRUPOS = {
    'preta':    ['pelagem_preta', 'pelagem_preto', 'pelagem_negro'],
    'branca':   ['pelagem_branca'],
    'cinza':    ['pelagem_cinza', 'pelagem_cinza/branca'],
    'caramela': ['pelagem_caramela', 'pelagem_caramelo', 'pelagem_amarelo'],
    'marrom':   ['pelagem_marrom', 'pelagem_marrón'],
    'rajada':   ['pelagem_rajada', 'pelagem_rajado'],
    'tricolor': ['pelagem_tricolor', 'pelagem_branca,preta,caramela'],
}

def recomendar(tipo, preferencias, n=5):
    """
    tipo: 'cachorro' ou 'gato'
    preferencias: dict — veja tabela acima
    n: número de resultados
    """
    if tipo == 'cachorro':
        df_orig, df_pca, pca, kmeans, scaler = orig_c, df_pca_cachorro, pca_cachorro, kmeans_cachorro, scaler_c
    else:
        df_orig, df_pca, pca, kmeans, scaler = orig_g, df_pca_gato, pca_gato, kmeans_gato, scaler_g

    vec = {c: 0.0 for c in feat_cols}

    vec['sexo']  = {'macho': 1.0, 'fêmea': 0.0}.get(preferencias.get('sexo', ''), 0.5)
    vec['porte'] = {'pequeno': 0.0, 'médio': 1.0, 'grande': 2.0}.get(preferencias.get('porte', ''), 1.0)
    vec['idade'] = {'filhote': 0.5, 'jovem': 2.0, 'adulto': 4.5, 'idoso': 7.0}.get(
        preferencias.get('idade', ''), 4.0)

    for col in PELAGEM_GRUPOS.get(preferencias.get('pelagem', ''), []):
        if col in vec: vec[col] = 1.0

    for col in TEMPERAMENTO_GRUPOS.get(preferencias.get('temperamento', ''), []):
        if col in vec: vec[col] = 1.0

    for c in preferencias.get('cuidados', []):
        key = f'cuidados_{c}'
        if key in vec: vec[key] = 1.0

    if preferencias.get('necessidades_especiais'):
        key = 'cuidados_precisa de cuidados especiais'
        if key in vec: vec[key] = 1.0

    moradia = preferencias.get('moradia', 'ambos')
    if moradia == 'apartamento':
        if 'vive_apartamento' in vec: vec['vive_apartamento'] = 1.0
    elif moradia == 'casa com quintal':
        if 'vive_casa com quintal' in vec: vec['vive_casa com quintal'] = 1.0
    else:
        for k in ['vive_apartamento', 'vive_casa com quintal']:
            if k in vec: vec[k] = 0.5

    if preferencias.get('criancas') and 'sociavel_crianças' in vec:
        vec['sociavel_crianças'] = 1.0

    outros_pets = preferencias.get('outros_pets', [])
    if 'cachorros' in outros_pets and 'sociavel_cachorros' in vec:
        vec['sociavel_cachorros'] = 1.0
    if 'gatos' in outros_pets and 'sociavel_gatos' in vec:
        vec['sociavel_gatos'] = 1.0

    arr        = np.array([[vec[c] for c in feat_cols]])
    arr_scaled = scaler.transform(arr)
    arr_pca    = pca.transform(arr_scaled)
    cluster    = int(kmeans.predict(arr_pca)[0])

    pc_cols  = [c for c in df_pca.columns if c != 'cluster']
    mask     = df_pca['cluster'] == cluster
    sims     = cosine_similarity(arr_pca, df_pca[mask][pc_cols].values)[0]
    top_idx  = np.argsort(sims)[::-1][:n]
    idx_list = df_pca[mask].index.tolist()

    print(f'Cluster encontrado: {cluster} (grupo {cluster + 1})')
    print(f'{"─"*70}')
    resultados = []
    for rank, i in enumerate(top_idx, 1):
        row = df_orig.iloc[idx_list[i]]
        nome = str(row.get('nome', '')) if pd.notna(row.get('nome')) else '(sem nome)'
        raca = str(row.get('raca', '')) if pd.notna(row.get('raca')) else ''
        tem_img = str(row.get('imagem', '')).startswith('http')
        print(f'  #{rank} [{sims[i]:.3f}] {nome} | {raca} | {row.sexo} | {row.porte} | '
              f'{row.idade} | pelagem={row.pelagem} | temperamento={row.temperamento}')
        print(f'       sociavel={row.sociavel_com} | cuidados={row.cuidados_veterinarios} | '
              f'imagem={"✅" if tem_img else "❌"}')
        resultados.append(df_orig.iloc[idx_list[i]])
    return cluster, resultados


# ── Exemplo de uso ──
recomendar('cachorro', {
    'sexo': 'fêmea', 'porte': 'pequeno', 'idade': 'filhote',
    'pelagem': 'caramela', 'temperamento': 'brincalhão',
    'moradia': 'apartamento', 'criancas': True,
    'outros_pets': [], 'cuidados': ['vacinado', 'castrado'],
    'necessidades_especiais': False
})

## 💾 Salvar modelos com joblib
Execute esta célula após treinar para gerar os arquivos `.pkl` usados pelo backend.

In [ ]:
import joblib, os

PKL_DIR = os.path.join('backend', 'model', 'pkl')
os.makedirs(PKL_DIR, exist_ok=True)

joblib.dump(scaler_c,         os.path.join(PKL_DIR, 'scaler_cachorro.pkl'))
joblib.dump(scaler_g,         os.path.join(PKL_DIR, 'scaler_gato.pkl'))
joblib.dump(pca_cachorro,     os.path.join(PKL_DIR, 'pca_cachorro.pkl'))
joblib.dump(pca_gato,         os.path.join(PKL_DIR, 'pca_gato.pkl'))
joblib.dump(kmeans_cachorro,  os.path.join(PKL_DIR, 'kmeans_cachorro.pkl'))
joblib.dump(kmeans_gato,      os.path.join(PKL_DIR, 'kmeans_gato.pkl'))
joblib.dump(df_pca_cachorro,  os.path.join(PKL_DIR, 'df_pca_cachorro.pkl'))
joblib.dump(df_pca_gato,      os.path.join(PKL_DIR, 'df_pca_gato.pkl'))
joblib.dump(orig_c,           os.path.join(PKL_DIR, 'orig_cachorro.pkl'))
joblib.dump(orig_g,           os.path.join(PKL_DIR, 'orig_gato.pkl'))
joblib.dump(feat_cols,        os.path.join(PKL_DIR, 'feat_cols.pkl'))

print('✅ Modelos salvos em:', PKL_DIR)
for f in sorted(os.listdir(PKL_DIR)):
    size = os.path.getsize(os.path.join(PKL_DIR, f)) / 1024
    print(f'   {f:35s} {size:7.1f} KB')